# Step 12 — Grassmann angle metrics (metric v3)

Interactive demo of `frames/linalg/angles.py` and
`EvaluationHarness.concept_angles`: gauge-invariant, bounded,
scale-invariant angles/distances between a text's hidden states and a
concept's span. Synthetic sanity checks run on CPU; the last section needs
a GPU + the AWQ Llama weights.


In [ ]:
import math

import torch

from frames.linalg.angles import (
    grassmann_distance,
    mean_canonical_correlation,
    principal_angles,
    text_frame,
    vector_subspace_cosine,
)

D, K = 8, 3
g = torch.Generator().manual_seed(0)
x, _ = torch.linalg.qr(torch.randn(D, K, generator=g, dtype=torch.float64))
y, _ = torch.linalg.qr(torch.randn(D, K, generator=g, dtype=torch.float64))

print("self:      angles", principal_angles(x, x).tolist())
print("random:    angles", [round(a, 3) for a in principal_angles(x, y).tolist()])
print("distance x,y:", float(grassmann_distance(x, y)))
print("mcc x,y:     ", float(mean_canonical_correlation(x, y)))

# gauge invariance: right-rotating y changes nothing
r, _ = torch.linalg.qr(torch.randn(K, K, generator=g, dtype=torch.float64))
print("distance x,yR:", float(grassmann_distance(x, y @ r)))

# a "text" whose positions live in span(x) is at distance ~0
coeffs = torch.randn(40, K, generator=g, dtype=torch.float64)
h = coeffs @ x.mT
print("low-rank text distance to x:", float(grassmann_distance(text_frame(h, K), x)))
print("token cosines (all 1):", vector_subspace_cosine(h[:3], x).tolist())

# random-direction calibration: cos ~ sqrt(K/D)
rand = torch.randn(2000, D, generator=g, dtype=torch.float64)
print(
    f"random-vector mean cos {float(vector_subspace_cosine(rand, x).mean()):.3f}"
    f"  vs sqrt(K/D) {math.sqrt(K / D):.3f}"
)

In [ ]:
# GPU section: real concepts and texts (AWQ Llama)
from frames.evaluation.harness import EvaluationHarness
from frames.representations import FrameUnembeddingRepresentation

fur = FrameUnembeddingRepresentation.from_model_id(
    "hugging-quants/Meta-Llama-3.1-8B-Instruct-AWQ-INT4", device_map="cuda:0"
)
harness = EvaluationHarness(
    fur, "/tmp/step12_angles.jsonl", min_lemmas_per_synset=11, max_token_count=3
)

sadness = fur.get_concept_cached("sadness.n.01", 11, 3)
music = fur.get_concept_cached("music.n.01", 11, 3)
print("rho(sadness, music):        ", float(sadness.rho(music)))
print("grassmann dist:", float(grassmann_distance(sadness.tensor[0], music.tensor[0])))
print(
    "mcc:           ",
    float(mean_canonical_correlation(sadness.tensor[0], music.tensor[0])),
)

texts = {
    "sad": "The old man wept alone in the empty house, grief pressing on him.",
    "neutral": "The committee will meet on Tuesday to review the quarterly budget.",
}
for label, text in texts.items():
    a = harness.concept_angles(text, sadness)
    print(f"{label:8s} vs sadness: {a}")